In [1]:
from sage.all import *
import numpy as np

F2 = GF(2)
R.<x> = PolynomialRing(F2)

print("Begin CRT modular polynomial calculation...")

def generate_mod_polynomials(d, count,u):
    mod_polys=[]
    if d>0:
        mod_polys = []
        c = 0
        max_attempts = 1000  

        for coeffs in range(2**(d)):  
            poly_coeffs = [0] * (d + 1)
            poly_coeffs[d] = 1 

            for i in range(d):
                poly_coeffs[i] = (coeffs >> i) & 1

            poly = sum(F2(poly_coeffs[i]) * x^i for i in range(d + 1))

            if poly.is_irreducible():
                mod_polys.append(poly)
                c += 1
                    
            max_attempts -= 1
            if max_attempts <= 0:
                break
    mod_polys_u=[]
    polys=[]
    for i in range(count):
        if isinstance(u, list):
            mod_polys_u.append([mod_polys[i], u[i]])
            polys.append(mod_polys[i])
        else:
            mod_polys_u.append([mod_polys[i], u])
            polys.append(mod_polys[i])
    return mod_polys_u,polys

Begin CRT modular polynomial calculation...


In [2]:
import pickle,re
F2=GF(2)
file_path = "EVP_571.txt"
lines_P = []
with open(file_path, "r", encoding="utf-8") as file:
    for line in file:
        lines_P.append(line.strip())
lines2=[]
for i, line in enumerate(lines_P):
    matches = re.findall(r'\d+', line)  
    result_p = []
    for match in matches:
        if not match.strip():
            result_p.append([])
        else:
            elements = [elem.strip() for elem in match.split(',')]
            result_p.append(elements)
    lines2.append(result_p)
def string_to_vector(vector):
    if(vector=='1'):
        poly=F2(1)
    elif(vector=='0'):
        poly=F2(0)
    return poly
G_ld=[]
for i, line in enumerate(lines2):
    list1=[]
    for j in range(0, 6):
        list1.extend(line[j][0])
    G_ld.append(list1)
G_matrix = Matrix(F2, G_ld).transpose() 
print(G_matrix)
with open(f"EVP_571.pkl", "wb") as f:
        pickle.dump(G_matrix, f)

[0 0 1 0 0 1 0 0 1 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 1 1 0 0 1 0 1 1 1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 1 1 1 0 1 1 0 0 1 0 0 0 1 0 0 0 0 1 1 0 0 1 0 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 1 0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 1 1 0 0 1 0 1 1 1 0 0 1 1 1 1 0 1 0 1 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 1 0 0 0 0 0 0 1 1 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 1 1 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1

In [3]:
def compute_coefficients_matrix(mod_polys_u, max_degree,u,BD,n):
    matrix_rows = []
    print(max_degree)
    for i in range(max_degree + 1):
        row = []
        x_power=BD[i]
        for [poly,u_val] in mod_polys_u:
            coeffs=g_adic_expansion(x_power, poly, u_val)
            row.extend(coeffs)
        
        coeffs = x_power.list()
        if max_degree <= n-1:
            coeffs += [0] * (n - len(coeffs))
            extra_vector = coeffs[-u[-1]:]
        else:
            coeffs += [0] * (2*n-1 - len(coeffs))
            extra_vector = coeffs[-u[-1]:]
        
        row.extend(extra_vector)
        matrix_rows.append(row)
    
    return matrix(F2, matrix_rows).transpose()

In [4]:
import pickle
def compute_high_degree_matrix(BD,n):
    mod_poly=x^571 + x^10 + x^5 + x^2 + 1

    F2_n = R.quotient(mod_poly, 'z')
    z = F2_n.gen()

    B_cols = []
    for poly in BD:
        poly_mod = F2_n(poly)
        coeffs = poly_mod.list()
        coeffs += [0] * (n - len(coeffs))
        B_cols.append(vector(F2, coeffs))
    B = matrix(F2, B_cols).transpose()  

    matrix_cols = []
    for i in range(n, 2*n-1):
        xi_mod = F2_n(x^i)
        v = vector(F2, xi_mod.list() + [0]*(n - len(xi_mod.list())))
        c = B.solve_right(v)
        matrix_cols.append(c)

    C = matrix(F2, matrix_cols).transpose()

    identity_matrix = matrix.identity(F2, n)
    extended_matrix = identity_matrix.augment(C)
    return extended_matrix

In [5]:
import numpy as np

def save_matrices(TD,T2D, C, filename_prefix="matrices"):
    with open("EVP_571.pkl", "rb") as f:
        EVP = pickle.load(f)
    TD[10:16, :] = EVP[:, :571]  
    T2D[10:16]=EVP
    CT2D_inv = C * T2D.inverse()
    with open(f"{filename_prefix}_TD_pn.pkl", "wb") as f:
        pickle.dump(TD, f)
    with open(f"{filename_prefix}_CT2D_inv_pn.pkl", "wb") as f:
        pickle.dump(CT2D_inv, f)
    with open(f"{filename_prefix}_T2D_inv_pn.pkl", "wb") as f:
        pickle.dump(T2D.inverse(), f)

In [6]:
def g_adic_expansion(f_poly, g_poly, num_terms):
    R = g_poly.parent()
    d = g_poly.degree()
    expansion_coeffs = []
    current_f = f_poly
    for _ in range(num_terms):
        remainder = current_f % g_poly
        remainder_coeffs = remainder.list()
        padding_length = d - len(remainder_coeffs)
        if padding_length > 0:
            remainder_coeffs.extend([0] * padding_length)
        expansion_coeffs.extend(remainder_coeffs)
        current_f = (current_f - remainder) // g_poly
    return expansion_coeffs

In [7]:
from sage.all import GF, PolynomialRing

F2 = GF(2)
R.<x> = PolynomialRing(F2)
n=571
u=[5, 3, 1, 1, 1, 1, 1, 1, 0, 1, 4]
num=[2, 1, 2, 3, 6, 9, 17, 30, 0, 66]
all_mod_polys_u=[]
t=10
allpolys=[]
for i in range(t):
    mod_polys_u,polys=generate_mod_polynomials(i+1, num[i],u[i])
    all_mod_polys_u+= mod_polys_u
    allpolys+=polys
BD = []

with open("2^571_l(2D)_pn_basis.txt", "r", encoding="utf-8") as f:
    for line in f:
        poly_str = line.strip()
        if poly_str:  
            poly = R(poly_str)
            BD.append(poly)
print(f"A total of {len(BD)} polynomials were read.")
TD = compute_coefficients_matrix(all_mod_polys_u, n-1,u,BD,n)
T2D = compute_coefficients_matrix(all_mod_polys_u, 2*n-2,u,BD,n)
C = compute_high_degree_matrix(BD[:571],n)
# save_matrices(TD,T2D, C, filename_prefix=n)

A total of 1141 polynomials were read.
570
1140


In [8]:
from sage.all import Matrix, GF

F2 = GF(2)
n = 571
B_cols = []
indices = []
for idx, poly in enumerate(BD):
    coeffs = poly.list() + [0]*(n - len(poly.list()))
    B_cols.append(coeffs)
    if len(B_cols) == n:
        B = Matrix(F2, B_cols).transpose()
        if B.rank() == n:
            indices = list(range(idx - n + 1, idx + 1))
            break
        else:
            B_cols.pop(0)
if len(indices) < n:
    raise ValueError("Unable to find 2^33 linearly independent polynomials")
BD_basis = [BD[i] for i in indices]